
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>




# Schemas and Tables on Databricks
In this demonstration, you will create and explore schemas and tables.

## Learning Objectives
By the end of this lesson, you should be able to:

* Use Spark SQL DDL to define schemas and tables
* Differentiate between managed and external tables in Spark SQL 
* Explain how managed and external tables impact storage location and management



**Resources**
* <a href="https://docs.databricks.com/user-guide/tables.html" target="_blank">Schemas and Tables - Databricks Docs</a>
* <a href="https://docs.databricks.com/user-guide/tables.html#managed-and-unmanaged-tables" target="_blank">Managed and Unmanaged Tables</a>
* <a href="https://docs.databricks.com/user-guide/tables.html#create-a-table-using-the-ui" target="_blank">Creating a Table with the UI</a>
* <a href="https://docs.databricks.com/user-guide/tables.html#create-a-local-table" target="_blank">Create a Local Table</a>
* <a href="https://spark.apache.org/docs/latest/sql-data-sources-load-save-functions.html#saving-to-persistent-tables" target="_blank">Saving to Persistent Tables</a>



## Lesson Setup
The following script clears out previous runs of this demo and configures some Hive variables that will be used in our SQL queries.

In [0]:
%run ./Includes/Classroom-Setup-03.1


 
## Schemas
Let's start by creating a schema (database).

In [0]:
CREATE SCHEMA IF NOT EXISTS ${da.schema_name}_default_location;



 
Note that the location of the first schema (database) is in the default location under **`dbfs:/user/hive/warehouse/`** and that the schema directory is the name of the schema with the **`.db`** extension

In [0]:
DESCRIBE SCHEMA EXTENDED ${da.schema_name}_default_location;

## Managed Tables

We will create a **managed** table (by not specifying a path for the location).

We will create the table in the schema (database) we created above.

Note that the table schema must be defined because there is no data from which to infer the table's columns and data types

In [0]:
USE ${da.schema_name}_default_location; 

CREATE OR REPLACE TABLE managed_table (width INT, length INT, height INT);
INSERT INTO managed_table 
VALUES (3, 2, 1);
SELECT * FROM managed_table;


 
We can look at the extended table description to find the location (you'll need to scroll down in the results).

In [0]:
DESCRIBE EXTENDED managed_table;



By default, **managed** tables in a schema without the location specified will be created in the **`dbfs:/user/hive/warehouse/<schema_name>.db/`** directory.

We can see that, as expected, the data and metadata for our table are stored in that location.

In [0]:
%python 
tbl_location = spark.sql(f"DESCRIBE DETAIL managed_table").first().location
print(tbl_location)

files = dbutils.fs.ls(tbl_location)
display(files)


 
Drop the table.

In [0]:
DROP TABLE managed_table;


 
Note the table's directory and its log and data files are deleted. Only the schema (database) directory remains.

In [0]:
%python 
schema_default_location = spark.sql(f"DESCRIBE SCHEMA {DA.schema_name}_default_location").collect()[3].database_description_value
print(schema_default_location)
dbutils.fs.ls(schema_default_location)


 
## External Tables
Next, we will create an **external** (unmanaged) table from sample data. 

The data we are going to use are in CSV format. We want to create a Delta table with a **`LOCATION`** provided in the directory of our choice.

In [0]:
USE ${da.schema_name}_default_location;

CREATE OR REPLACE TEMPORARY VIEW temp_delays USING CSV OPTIONS (
  path = '${da.paths.datasets}/flights/departuredelays.csv',
  header = "true",
  mode = "FAILFAST" -- abort file parsing with a RuntimeException if any malformed lines are encountered
);
CREATE OR REPLACE TABLE external_table LOCATION '${da.paths.working_dir}/external_table' AS
  SELECT * FROM temp_delays;

SELECT * FROM external_table;


 
Let's note the location of the table's data in this lesson's working directory.

In [0]:
DESCRIBE EXTENDED external_table;


 
Now, we drop the table.

In [0]:
DROP TABLE external_table;


 
The table definition no longer exists in the metastore, but the underlying data remain intact.

In [0]:
%python 
tbl_path = f"{DA.paths.working_dir}/external_table"
files = dbutils.fs.ls(tbl_path)
display(files)


## Clean up
Drop the schema.

In [0]:
DROP SCHEMA ${da.schema_name}_default_location CASCADE;


Run the following cell to delete the tables and files associated with this lesson.

In [0]:
%python 
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>